# Two-Body Orbital Propagation and First Integrals

This notebook demonstrates how Newton's inverse-square gravity law leads to the conserved quantities of Keplerian motion:

- **Specific orbital energy** \(\epsilon\)
- **Specific angular momentum** \(\mathbf{h} = \mathbf{r} \times \mathbf{v}\)

We propagate a spacecraft state under a pure two-body Earth model and verify that these first integrals remain nearly constant under high-accuracy numerical integration.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed to enable 3D projection)

# Gravitational parameter of Earth [km^3/s^2]
mu = 398600.4418

## 1) Two-body equations of motion

For the Cartesian state \(\mathbf{x} = [\mathbf{r}, \mathbf{v}]\), the two-body dynamics are

\[
\dot{\mathbf{r}} = \mathbf{v},\qquad
\dot{\mathbf{v}} = -\mu \frac{\mathbf{r}}{\|\mathbf{r}\|^3}.
\]

This acceleration is always central (points toward Earth), which is why angular momentum is conserved.

In [ ]:
def two_body_ode(t, state, mu):
    r = state[:3]
    v = state[3:]
    r_norm = np.linalg.norm(r)

    r_dot = v
    v_dot = -mu * r / r_norm**3

    return np.hstack((r_dot, v_dot))

## 2) Initial state and propagation setup

Given:

- \(\mathbf{r}_0 = [7000, 0, 0]\) km
- \(\mathbf{v}_0 = [0, 7.5, 1.0]\) km/s

We compute an approximate orbital period from semi-major axis estimated from initial specific energy, then propagate for **3 periods** with tight tolerances (`rtol=1e-12`, `atol=1e-12`).

In [ ]:
r0 = np.array([7000.0, 0.0, 0.0])  # km
v0 = np.array([0.0, 7.5, 1.0])        # km/s
x0 = np.hstack((r0, v0))

# Initial orbital energy and period estimate
r0_norm = np.linalg.norm(r0)
v0_norm = np.linalg.norm(v0)
epsilon0 = 0.5 * v0_norm**2 - mu / r0_norm

a0 = -mu / (2 * epsilon0)  # valid for bound ellipse (epsilon < 0)
T0 = 2 * np.pi * np.sqrt(a0**3 / mu)

t_span = (0.0, 3.0 * T0)
t_eval = np.linspace(t_span[0], t_span[1], 4000)

sol = solve_ivp(
    two_body_ode,
    t_span=t_span,
    y0=x0,
    args=(mu,),
    t_eval=t_eval,
    rtol=1e-12,
    atol=1e-12,
    method='DOP853'
)

if not sol.success:
    raise RuntimeError(sol.message)

r = sol.y[:3].T
v = sol.y[3:].T
t = sol.t

## 3) Compute first integrals and vis-viva consistency

At each time step we compute:

- \(\|\mathbf{r}\|\), \(\|\mathbf{v}\|\)
- Specific energy \(\epsilon = \frac{v^2}{2} - \mu/r\)
- Specific angular momentum vector \(\mathbf{h} = \mathbf{r}\times\mathbf{v}\) and \(\|\mathbf{h}\|\)
- Semi-major axis from energy: \(a = -\mu/(2\epsilon)\)
- Vis-viva prediction: \(v^2_{vv} = \mu\left(2/r - 1/a\right)\)
- Error: \(v^2 - v^2_{vv}\)

In [ ]:
r_norm = np.linalg.norm(r, axis=1)
v_norm = np.linalg.norm(v, axis=1)

epsilon = 0.5 * v_norm**2 - mu / r_norm
h_vec = np.cross(r, v)
h_norm = np.linalg.norm(h_vec, axis=1)

a_from_energy = -mu / (2 * epsilon)
v2_visviva = mu * (2.0 / r_norm - 1.0 / a_from_energy)
visviva_error = v_norm**2 - v2_visviva

# Eccentricity vector and magnitude
# e = (v x h)/mu - r/|r|
e_vec = np.cross(v, h_vec) / mu - r / r_norm[:, None]
e_mag = np.linalg.norm(e_vec, axis=1)

print(f"Initial specific energy: {epsilon[0]: .12f} km^2/s^2")
print(f"Final specific energy:   {epsilon[-1]: .12f} km^2/s^2")
print(f"Max |Δepsilon| over arc: {np.max(np.abs(epsilon - epsilon[0])): .3e} km^2/s^2")
print(f"Max |Δ|h|| over arc:     {np.max(np.abs(h_norm - h_norm[0])): .3e} km^2/s")
print(f"Mean eccentricity:       {np.mean(e_mag): .8f}")

## 4) Orbit classification from energy and eccentricity

Classification rules:

- **Elliptic**: \(\epsilon < 0\) and \(e < 1\)
- **Parabolic**: \(\epsilon \approx 0\) and \(e \approx 1\)
- **Hyperbolic**: \(\epsilon > 0\) and \(e > 1\)

In [ ]:
eps_mean = np.mean(epsilon)
e_mean = np.mean(e_mag)

def classify_orbit(epsilon_value, e_value, tol=1e-8):
    if epsilon_value > tol:
        return "Hyperbolic"
    if epsilon_value < -tol:
        return "Elliptic"
    # near zero energy
    return "Parabolic (near numerical tolerance)"

orbit_type = classify_orbit(eps_mean, e_mean)
print(f"Orbit type from energy sign: {orbit_type}")
print(f"Average epsilon: {eps_mean:.12f} km^2/s^2")
print(f"Average eccentricity: {e_mean:.8f}")

## 5) Plots

The following plots should show near-constant energy and angular momentum, with very small vis-viva residuals (numerical error).

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(r[:, 0], r[:, 1], r[:, 2], lw=1.5, label='Trajectory')
ax.scatter([0], [0], [0], color='tab:blue', s=80, label='Earth center')
ax.set_title('3D Two-Body Orbit')
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.legend()
ax.set_box_aspect((1, 1, 1))
plt.tight_layout()
plt.show()

fig, axs = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

axs[0].plot(t / 3600.0, epsilon, color='tab:green')
axs[0].set_ylabel('Specific energy [km²/s²]')
axs[0].set_title('Specific Orbital Energy vs Time')
axs[0].grid(True, alpha=0.3)

axs[1].plot(t / 3600.0, h_norm, color='tab:purple')
axs[1].set_ylabel('|h| [km²/s]')
axs[1].set_title('Angular Momentum Magnitude vs Time')
axs[1].grid(True, alpha=0.3)

axs[2].plot(t / 3600.0, visviva_error, color='tab:red')
axs[2].set_ylabel(r'$v^2 - v_{vis-viva}^2$ [km²/s²]')
axs[2].set_xlabel('Time [hours]')
axs[2].set_title('Vis-viva Equation Error')
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6) Increase initial velocity: parabolic and hyperbolic transitions

For fixed radius \(r_0\), energy depends on speed:
\[
\epsilon = \frac{v^2}{2} - \frac{\mu}{r_0}.
\]

The **escape speed** (parabolic threshold) is
\[
v_{esc} = \sqrt{2\mu/r_0},
\]
which gives \(\epsilon = 0\). Speeds above this yield \(\epsilon > 0\), i.e. a hyperbola.

In [ ]:
v0_mag = np.linalg.norm(v0)
v_esc = np.sqrt(2 * mu / r0_norm)
scale_parabolic = v_esc / v0_mag

v_parabolic = v0 * scale_parabolic
v_hyperbolic = v0 * (1.10 * scale_parabolic)  # 10% above escape threshold

def energy_and_ecc(r_init, v_init, mu):
    rmag = np.linalg.norm(r_init)
    vmag = np.linalg.norm(v_init)
    eps = 0.5 * vmag**2 - mu / rmag
    h = np.cross(r_init, v_init)
    e = np.cross(v_init, h) / mu - r_init / rmag
    return eps, np.linalg.norm(e)

eps_base, e_base = energy_and_ecc(r0, v0, mu)
eps_para, e_para = energy_and_ecc(r0, v_parabolic, mu)
eps_hyp, e_hyp = energy_and_ecc(r0, v_hyperbolic, mu)

print(f"Baseline speed:   {v0_mag:.6f} km/s -> epsilon={eps_base:.6f}, e={e_base:.6f}")
print(f"Escape speed:     {v_esc:.6f} km/s -> epsilon={eps_para:.6e}, e={e_para:.6f}")
print(f"Hyperbolic speed: {np.linalg.norm(v_hyperbolic):.6f} km/s -> epsilon={eps_hyp:.6f}, e={e_hyp:.6f}")

In [ ]:
# Propagate parabolic-threshold and hyperbolic cases for comparison

def propagate_case(v_init, t_final):
    state0 = np.hstack((r0, v_init))
    te = np.linspace(0, t_final, 2000)
    sol_case = solve_ivp(
        two_body_ode,
        (0, t_final),
        state0,
        args=(mu,),
        t_eval=te,
        rtol=1e-12,
        atol=1e-12,
        method='DOP853'
    )
    return sol_case.y[:3].T

r_para = propagate_case(v_parabolic, 4 * 3600.0)
r_hyp = propagate_case(v_hyperbolic, 4 * 3600.0)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(r[:,0], r[:,1], r[:,2], label='Baseline (Elliptic)', lw=1.5)
ax.plot(r_para[:,0], r_para[:,1], r_para[:,2], label='Parabolic threshold', lw=1.5)
ax.plot(r_hyp[:,0], r_hyp[:,1], r_hyp[:,2], label='Hyperbolic', lw=1.5)
ax.scatter([0], [0], [0], s=80, color='tab:blue')
ax.set_title('Orbit Type Transition by Increasing Initial Speed')
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.legend()
ax.set_box_aspect((1, 1, 1))
plt.tight_layout()
plt.show()

## 7) Physical interpretation

- **Specific energy \(\epsilon\)** combines kinetic and gravitational potential energy per unit mass. In an ideal two-body model, it is conserved.
- **Specific angular momentum \(\mathbf{h}\)** encodes the orbit plane and areal sweep rate. Its conservation reflects zero external torque in a central force field.
- **Numerical drift** (tiny changes in these constants) comes from finite precision and truncation error in the numerical integrator. Tight tolerances reduce, but never completely remove, this drift.
- **Why energy defines orbit type**: the sign of total specific energy determines whether the object is bound to Earth (negative) or unbound (zero/positive).

## 8) Summary

Newton's law of gravitation directly generates trajectories that obey Kepler-like behavior: planar motion, conservation of areal velocity (via constant angular momentum), and conic-section orbit families set by specific energy. The numerical propagation confirms these first integrals to high precision and shows how increasing initial speed transitions the orbit from elliptic to parabolic to hyperbolic.